# 11 · Cascade: ASR → VQA (T1_Speech)

Whisper transcripts replace the text question as input to the T2_Text VLM. Model, prompt, generation
settings and ids are the same as in `t2_text/run_vlm.py`, so the only difference from T2_Text is where
the question comes from. The T2_Text twin is generated in the same session (same GPU and library versions).

Kaggle or Colab: GPU T4, Internet On, secret `HF_TOKEN`. Scoring uses the official scorer (torch >= 2.6),
which runs here but not on an Intel Mac. `12_speech_vs_text` then runs locally on the downloaded files.

## Setup

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
!git clone https://github.com/mohinska/mmculture-qa.git
%cd mmculture-qa
!pip install -q -r requirements.txt
%env MMCQA_OUT=/kaggle/working

In [ ]:
!python shared/download_data.py dev --task sqa

## 1. Transcripts
`asr/` is not in the repo (data licence), so the transcripts are rebuilt here with the same model and backend as `10_asr_whisper`.

In [ ]:
!python t1_speech/run_asr.py --split dev

## 2. Cascade and its T2_Text twin
Same `--model`, `--prompt-id` and `--n` for both runs.

In [ ]:
MODEL, PROMPT_ID, N = "Qwen/Qwen3-VL-4B-Instruct", "gold_style_v1", 200

!python t1_speech/run_cascade.py --model {MODEL} --prompt-id {PROMPT_ID} --n {N}
!python t2_text/run_vlm.py --model {MODEL} --prompt-id {PROMPT_ID} --n {N}

## 3. Scoring
Official scorer through `evaluate()`: writes `results/<preds name>.json` and `<preds>.scores.jsonl`.

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared").is_dir())
sys.path[:0] = [str(ROOT / "shared"), str(ROOT / "t1_speech")]

from data import load_jsonl, preds_path
from metrics import evaluate
from speech import ASR_SHORT, DEFAULT_ASR

VLM = MODEL.split("/")[-1]
SPEECH = preds_path("t1_speech_cascade", "dev", f"{ASR_SHORT[DEFAULT_ASR]}+{VLM}", PROMPT_ID)
TEXT = preds_path("t2_text", "dev", VLM, PROMPT_ID)

a, b = ({r["id"] for r in load_jsonl(p)} for p in (SPEECH, TEXT))
assert a == b, f"speech {len(a)} ids, text {len(b)}, common {len(a & b)}"
for p in (SPEECH, TEXT):
    res = evaluate(p, "dev")
    print(p.name, {k: res[k] for k in ("n", "bertscore_f1", "bleu", "rouge_l", "coverage")})

## Save results

Then: right panel > Output > the zip > Download.

In [ ]:
!cd /kaggle/working && zip -r t1_cascade_$(date +%m%d_%H%M).zip asr preds mmculture-qa/results